# qust Overview：从安装到核心思路

[项目地址](https://baiguoname.github.io/qust/site) · [git地址](https://github.com/baiguoname/qust)


本节从安装、核心对象和常见工作流开始，系统说明 qust 的使用方式：

1. 怎么安装和启动；
2. qust 里的 `Expr`、数据、上下文、runtime 分别是什么；
3. 为什么它适合量化研究、回测和实时计算；
4. 常见算子怎么写；
5. 什么时候用 `calc_data`，什么时候用流式 `runtime().calc_stream(...)`；
6. monitor 图表怎么在 Jupyter 里直接显示。

你可以把 qust 理解成一个“Python 写表达式、Rust 执行计算、金融场景有现成积木”的量化研究框架。Python 端负责让策略和因子写起来像公式；底层执行器负责把这些公式编译成高性能计算流程。


## 1. 安装

最常见的安装方式：

```bash
pip install qust
```

如果你需要做参数优化，通常还会用到 Optuna：

```bash
pip install optuna
```

如果你要连接 ClickHouse，需要确认服务器已经开启对应端口，并在 Python 端配置 datasource。qust 自身提供 `qds.clickhouse(...)` / `ChConfig(...).as_datasource(...)` 这类入口；真实连接示例会在后面用代码块说明，但本节不依赖外部数据库，保证直接运行。

示例代码直接使用已安装的 `qust` 包；运行前请先执行 `pip install qust`。


In [1]:
import qust as qs
import qust.datasource as qds
import qust.future.future  # 注册 future / ta / bt / stra 等命名空间
from qust import col
from qust._polars import pl

pl.Config.set_tbl_rows(16)
pl.Config.set_tbl_cols(20)

KLINE_PATH = "https://github.com/baiguoname/qust/blob/main/examples/data/data_kline3.parquet?raw=true"
TICK_PATH = "https://github.com/baiguoname/qust/blob/main/examples/data/data_tick.parquet?raw=true"


## 2. qust 的核心心智模型

qust 的核心对象是 `Expr`。`Expr` 只描述“怎么算”，不直接保存数据。数据可以来自：

- `polars.DataFrame`：研究阶段最常见；
- parquet/csv/lazy 数据源：适合分批读取；
- ClickHouse：适合连接已有行情库或因子库；
- Python UDF：当某些逻辑暂时没有原生算子时，用 UDF 补上。

一条计算通常长这样：

```python
expr = col("close").mean().rolling(20).over("ticker", "ct")
result = expr.calc_data(data)
```

这行表达的是：对每个 `ticker + ct` 合约独立维护状态，在每个合约内部计算 `close` 的 20 窗口均值。

更完整的流程可以写成：

```text
数据源 -> Expr 编译 -> Exec 执行器 -> DataFrame / Stream / Monitor
```

这也是 qust 和直接写循环最大的差别：你写的是表达式图，qust 决定怎么执行、怎么复用状态、怎么在 group/rolling/stream 下维护上下文。


## 3. 读取 GitHub 样例数据

这里从 GitHub 样例数据读取 K 线文件。字段含义：

| 字段 | 含义 |
| --- | --- |
| `ticker` | 品种代码 |
| `datetime` | K 线时间 |
| `open/high/low/close` | OHLC 价格 |
| `volume` | 成交量 |
| `is_finished` | 当前 K 线是否完成 |

这个数据集有 60 多万行，足够展示普通 DataFrame 计算、分组计算、流式计算和画图。


In [2]:
data_kline = pl.read_parquet(KLINE_PATH).sort(["ticker", "ct", "datetime"])
PLOT_ROW = data_kline.select("ticker", "ct").unique().sort(["ticker", "ct"]).row(0, named=True)
PLOT_TICKER = PLOT_ROW["ticker"]
PLOT_CT = PLOT_ROW["ct"]

print("shape:", data_kline.shape)
print("tickers:", data_kline.select(pl.col("ticker").unique().sort()).to_series().to_list())
print("plot contract:", PLOT_TICKER, PLOT_CT)
data_kline.head(8)


shape: (408782, 8)
tickers: ['AP', 'RM', 'SA', 'al', 'eb', 'eg', 'fu', 'rb']
plot contract: AP 205


ticker,ct,datetime,open,high,low,close,volume
str,i32,datetime[ms],f64,f64,f64,f64,f64
"""AP""",205,2022-01-04 09:00:00,8394.0,8394.0,8392.0,8392.0,1100.0
"""AP""",205,2022-01-04 09:05:00,8385.0,8389.0,8348.0,8378.0,11169.0
"""AP""",205,2022-01-04 09:10:00,8375.0,8376.0,8298.0,8302.0,14001.0
"""AP""",205,2022-01-04 09:15:00,8301.0,8315.0,8271.0,8280.0,12839.0
"""AP""",205,2022-01-04 09:20:00,8279.0,8285.0,8243.0,8246.0,11496.0
"""AP""",205,2022-01-04 09:25:00,8243.0,8267.0,8220.0,8267.0,12307.0
"""AP""",205,2022-01-04 09:30:00,8267.0,8325.0,8252.0,8323.0,10889.0
"""AP""",205,2022-01-04 09:35:00,8323.0,8405.0,8315.0,8371.0,19223.0


## 4. 最基础的表达式：选择列、派生列、调用 Polars 表达式

`col(...)` 是 qust 最常用的入口。

常见写法：

| 写法 | 含义 |
| --- | --- |
| `col("close")` | 选择一列 |
| `col("open", "close")` | 选择多列 |
| `col.lit(1.0)` | 字面量 |
| `col("close") / col("open") - col.lit(1.0)` | qust 表达式运算 |
| `col(pl.col("close").log().alias("log_close"))` | 直接嵌入 Polars 表达式 |

这点很重要：qust 不是要替代 Polars 的每个表达式，而是在 Polars 擅长的列计算之上，加上自己的状态上下文、金融算子、流式执行和 monitor UI。


In [3]:
basic_expr = col(
    "ticker",
    "datetime",
    "open",
    "close",
    (col("close") / col("open") - col.lit(1.0)).alias("intrabar_ret"),
    col(pl.col("close").log().alias("log_close")),
)

basic_result = basic_expr.calc_data(data_kline.head(10))
basic_result


ticker,datetime,open,close,intrabar_ret,log_close
str,datetime[ms],f64,f64,f64,f64
"""AP""",2022-01-04 09:00:00,8394.0,8392.0,-0.000238,9.035034
"""AP""",2022-01-04 09:05:00,8385.0,8378.0,-0.000835,9.033365
"""AP""",2022-01-04 09:10:00,8375.0,8302.0,-0.008716,9.024252
"""AP""",2022-01-04 09:15:00,8301.0,8280.0,-0.00253,9.021598
"""AP""",2022-01-04 09:20:00,8279.0,8246.0,-0.003986,9.017484
"""AP""",2022-01-04 09:25:00,8243.0,8267.0,0.002912,9.020027
"""AP""",2022-01-04 09:30:00,8267.0,8323.0,0.006774,9.026778
"""AP""",2022-01-04 09:35:00,8323.0,8371.0,0.005767,9.032529
"""AP""",2022-01-04 09:40:00,8374.0,8385.0,0.001314,9.0342


## 5. 上下文：rolling、expanding、over、group_by

很多量化逻辑不是“当前行单独计算”，而是要依赖历史或分组。qust 把这些语义做成上下文：

| 上下文 | 典型含义 | 示例 |
| --- | --- | --- |
| `rolling(n)` | 固定长度滑动窗口 | 20 根 K 线均线 |
| `expanding()` | 从开始到当前的累积状态 | 累计收益、运行均值 |
| `over("ticker")` | 每个品种独立计算 | 每个合约单独滚动均线 |
| `group_by(...)` | 按 key 聚合 | 每日汇总、横截面聚合 |

下面先生成每个品种内部的 20 窗口均线、滚动波动率和运行均值。


In [4]:
feature_expr = col(
    "ticker",
    "ct",
    "datetime",
    "close",
    col("close").mean().rolling(20).alias("ma20"),
    col("close").std().rolling(20).alias("std20"),
    col("close").mean().expanding().alias("running_mean"),
).over("ticker", "ct")

feature_result = feature_expr.calc_data(data_kline.head(30_000))
feature_result.filter((pl.col("ticker") == PLOT_TICKER) & (pl.col("ct") == PLOT_CT)).head(12)


ticker,ct,datetime,close,ma20,std20,running_mean
str,i32,datetime[ms],f64,f64,f64,f64
"""AP""",205,2022-01-04 09:00:00,8392.0,null,null,8392.0
"""AP""",205,2022-01-04 09:05:00,8378.0,null,null,8385.0
"""AP""",205,2022-01-04 09:10:00,8302.0,null,null,8357.333333
"""AP""",205,2022-01-04 09:15:00,8280.0,null,null,8338.0
"""AP""",205,2022-01-04 09:20:00,8246.0,null,null,8319.6
"""AP""",205,2022-01-04 09:25:00,8267.0,null,null,8310.833333
"""AP""",205,2022-01-04 09:30:00,8323.0,null,null,8312.571429
"""AP""",205,2022-01-04 09:35:00,8371.0,null,null,8319.875
"""AP""",205,2022-01-04 09:40:00,8385.0,null,null,8327.111111


`group_by` 用于把多行压成分组结果。下面把分钟 K 线按 `date + ticker` 汇总，得到每天每个品种的最后收盘价和总成交量。


In [5]:
daily_expr = col(
    col("close").last_value().alias("last_close"),
    col("volume").sum().alias("volume"),
).group_by(
    col("datetime").dt.date().alias("date"),
    "ticker",
    "ct",
)

daily_by_ticker = daily_expr.calc_data(data_kline)
daily_by_ticker.head(12)


date,ticker,ct,last_close,volume
date,str,i32,f64,f64
2022-01-04,"""AP""",205,8164.0,296870.0
2022-01-05,"""AP""",205,8027.0,320800.0
2022-01-06,"""AP""",205,8219.0,212764.0
2022-01-07,"""AP""",205,8623.0,338351.0
2022-01-10,"""AP""",205,8634.0,133660.0
2022-01-11,"""AP""",205,8578.0,138517.0
2022-01-12,"""AP""",205,8479.0,151540.0
2022-01-13,"""AP""",205,8418.0,142607.0
2022-01-14,"""AP""",205,8538.0,170271.0


## 6. monitor：把结果直接画出来

qust 的 monitor 是交互式图表 UI。表达式仍然是 qust 写法：

```python
col("datetime", "close").monitor("slot_id").line().runtime().plot(data)
```

这里的关键点：

1. `monitor.line()` 只是把当前表达式变成绘图表达式；
2. `.runtime()` 表示进入可运行的 DataFrame runtime；
3. `.plot(..., open_in_jupyter=True)` 会在 输出一个真实 iframe，不是静态 PNG。


In [49]:
contract_price = (
    data_kline
    .filter((pl.col("ticker") == PLOT_TICKER) & (pl.col("ct") == PLOT_CT))
    .select("datetime", "close")
    .head(8_000)
)

price_runtime = col("datetime", "close").monitor("contract_close", show_axis_label=True).line().runtime()
price_runtime.plot(contract_price, open_in_jupyter=True, auto_open=False, height=420)


In [50]:
contract_daily_volume = (
    daily_by_ticker
    .filter((pl.col("ticker") == PLOT_TICKER) & (pl.col("ct") == PLOT_CT))
    .select("date", "volume")
    .sort("date")
)

volume_runtime = col("date", "volume").monitor("contract_daily_volume", show_axis_label=True).bar().runtime()
volume_runtime.plot(contract_daily_volume, open_in_jupyter=True, auto_open=False, height=420)


## 7. `calc_data` 和流式 `calc_stream` 的区别

很多新用户会被 `calc_data`、`runtime()`、`calc_stream` 搞混。可以按下面理解：

| 方法 | 输入 | 适用场景 |
| --- | --- | --- |
| `expr.calc_data(dataframe)` | 一个已经在内存里的 DataFrame | 小中型研究、调试、单次计算 |
| `expr.runtime().calc_stream(datasource)` | 一个分批数据源 | 大文件、实时行情、需要按 batch 持续更新状态 |
| `expr.runtime().plot(data)` | 数据或数据源 | 计算后直接进入 monitor UI |

流式不是简单地“分块 collect 后拼起来”。关键在于：`rolling`、`expanding`、`over` 这类有状态算子会把状态留在执行器里，下一批数据到来时继续更新。

下面用同一个运行均值表达式，分别对 DataFrame 和 parquet 数据源计算。


In [6]:
running_mean_expr = col("close").mean().expanding().alias("running_mean_close")

calc_data_tail = running_mean_expr.calc_data(data_kline.select("close")).tail(5)

stream_source = qds.from_dataframe(data_kline.select("close"), chunk_size=100_000)
calc_stream_tail = running_mean_expr.runtime().calc_stream(stream_source).tail(5)

print("calc_data tail:")
print(calc_data_tail)
print()
print("calc_stream tail:")
print(calc_stream_tail)


calc_data tail:
shape: (5, 1)
┌────────────────────┐
│ running_mean_close │
│ ---                │
│ f64                │
╞════════════════════╡
│ 7052.088246        │
│ 7052.079106        │
│ 7052.069952        │
│ 7052.06079         │
│ 7052.051634        │
└────────────────────┘

calc_stream tail:
shape: (5, 1)
┌────────────────────┐
│ running_mean_close │
│ ---                │
│ f64                │
╞════════════════════╡
│ 7052.088246        │
│ 7052.079106        │
│ 7052.069952        │
│ 7052.06079         │
│ 7052.051634        │
└────────────────────┘


## 8. 数据源：DataFrame、LazyFrame、parquet、ClickHouse

研究阶段通常从 `pl.read_parquet(...)` 开始；数据变大以后，可以切到 `qds.read_parquet(...)` 或 `qds.from_lazy(...)`，让 qust runtime 分批消费。

ClickHouse 的典型写法如下。本节不直接连接数据库，所以这里作为模板保留：

```python
import datetime as dt
import qust.datasource as qds
from qust import col

chdb = qds.clickhouse(
    url="192.168.1.139:9000",
    database="future_market_data",
    username="bg",
    password="***",
    query="select * from gq_tick where trading_date >= '2024-02-01'",
)

expr = col.filter(
    (col("trading_date") >= col.lit(dt.date(2024, 2, 1)))
).select(
    "code",
    "datetime",
    "last",
)

result = expr.runtime().calc_stream(chdb)
```

如果数据源支持谓词下推，过滤条件越靠近数据源越好；但具体能不能下推取决于 datasource 的实现、表达式是否能翻译成底层查询、以及文件/数据库本身的统计信息。


## 9. 为什么 qust 适合量化研究

qust 的优势主要不在“语法短”，而在这些地方：

1. **表达式可组合**：因子、信号、回测、统计、画图都能放进同一条表达式链。
2. **上下文清晰**：`rolling / expanding / over / group_by / batch` 明确告诉执行器状态怎么维护。
3. **流式友好**：同一个表达式可以先在历史数据上回测，再接流式数据做实时监控，减少研究和实盘之间的逻辑偏差。
4. **金融算子内置**：`ta`、`stra`、`bt`、`alpha`、`monitor`、`opt` 等命名空间把常见量化模块做成标准积木。
5. **可以接 Polars**：普通列计算仍然可以用 Polars 表达式，不需要为了用 qust 放弃 Polars 生态。
6. **高性能执行器**：核心执行路径在 Rust 里，适合批量跑因子、批量回测和多策略评估。
7. **交互分析**：monitor 可以把回测曲线、分布、热力图、K 线、散点等直接变成交互 UI。

下一份 portfolio 教程 会把这些积木串起来：用两条内置策略生成 PnL，做等权组合，再用 `convex_sr` 求组合权重并画出组合曲线。
